In [ ]:
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "neurocnl",
#     "snntorch",
# ]
# ///

# NeuroMorphic Pipeline — Snntorch Sim

Generated 2026-07-04 00:00 UTC.

**Architecture:** defined in the Architecture tab (CNL spec below).
**Pipeline config:** edit `config` in the next cell to change training parameters.

In [ ]:
# ── Pipeline configuration ───────────────────────────────────────────
# Workspace settings used when this notebook was generated.

config = {
    "dataset":   "",
    "framework": "snntorch_sim",
}

print('Config loaded:', config)

In [ ]:
import json as _json


def _nmtk_emit(
    epoch: int, total: int, loss: float, accuracy: float, layer_rates: dict
) -> None:
    print(
        _json.dumps(
            {
                "__nmtk_progress__": True,
                "epoch": epoch,
                "total_epochs": total,
                "loss": loss,
                "accuracy": accuracy,
                "layer_spike_rates": layer_rates,
            }
        ),
        flush=True,
    )


In [ ]:
import torch
from torch.utils.data import DataLoader
# No dataset selected — replace with your own DataLoader.
# train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
# test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)


## Architecture

Network compiled from CNL spec via NIR.

In [ ]:
# CNL spec — auto-generated from the Architecture canvas tab.
# To change the network, edit the Architecture tab and regenerate.
cnl_spec = '''
Define a network named graph.
# Network with 1 input, 2 hidden nodes, 1 output.
# flow: input → linear → lif → output

# Layers:
Define an input port named input with shape (1,).
Define a linear transformation named linear with weight matrix shape (1, 1).
Define a leaky integrate-and-fire neuron named lif with time constant 0.0025, resistance 1.0, leak voltage 0.0, and firing threshold 0.1.
Define an output port named output with shape (1,).

# Connections:
input connects to linear.
linear connects to lif.
lif connects to output.

'''

from neurocnl.compile import compile_to_nir

graph = compile_to_nir(cnl_spec)
print(f'Network: {len(graph.nodes)} nodes, {len(graph.edges)} edges')

In [ ]:
"""snnTorch network — auto-generated from NIR graph."""

import torch
import torch.nn as nn
import snntorch as snn
import numpy as np

_w = np.load('weights.npz')  # weights file saved alongside this notebook


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Linear layer: 'linear'  shape (1, 1)
        self.linear = nn.Linear(1, 1, bias=False)
        self.linear.weight.data = torch.from_numpy(_w['linear_weight'].copy())
        # LIF population: 'lif'  (1 neurons)
        self.lif = snn.Leaky(beta=0.960000, threshold=2.5000, reset_mechanism='zero', init_hidden=True, reset_delay=False)

    def forward(self, x):
        # initialise hidden states
        self.lif.init_leaky()
        # x: (T,B,C,H,W) time-first from tonic, or (B,C,H,W) for a single frame
        if x.dim() == 4:
            x = x.unsqueeze(0)  # (B,C,H,W) → (1,B,C,H,W)
        _x_seq = x
        spk_rec = []
        for t in range(_x_seq.shape[0]):
            x = _x_seq[t]
            x = self.linear(x)
            spk_lif = self.lif(x)
            x = spk_lif
            spk_rec.append(x)
        return torch.stack(spk_rec, dim=0), x  # (T, batch, out), last spk


net = Net().float()  # ponytail: npz weights load as float64; cast to match DataLoader float32 input
print(f'Net: {sum(p.numel() for p in net.parameters())} parameters')

In [ ]:
# ── Download as Python script ──────────────────────────────────
# Run this cell to download the notebook as a .py script.
import subprocess
subprocess.run(['jupyter', 'nbconvert', '--to', 'script',
                '__file__'], check=False)
print('Conversion triggered — check the file listing.')